# **FA 590- Statistical Learning in Finance**

# **Assignment 2: Loan Default and Return Prediction using Logistic Models**

**Ayush Sandip Parekh**

**Student ID: 20030220**

In [82]:
# Loading the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import r2_score
from imblearn.over_sampling import SMOTE
from collections import Counter

Dropping the 'id' Column

The id column is unique to each loan and does not contribute to predicting returns. Dropping it helps reduce noise.

In [83]:
loan = pd.read_csv('lc_loan.csv')
loan = loan.drop(columns=("id"))
print("Size of Dataset : " + str(loan.shape))

Size of Dataset : (933160, 36)


Checking Missing Values and Dataset Overview

Using loan.info() and loan.isnull().sum() helps identify missing values and understand the dataset structure, which is crucial before feature engineering.

In [84]:
loan.info()

loan.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 933160 entries, 0 to 933159
Data columns (total 36 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   loan_amnt                   933160 non-null  float64
 1   funded_amnt                 933160 non-null  float64
 2   int_rate                    933160 non-null  float64
 3   installment                 933160 non-null  float64
 4   grade                       933160 non-null  object 
 5   sub_grade                   933160 non-null  object 
 6   emp_length                  874392 non-null  object 
 7   home_ownership              933160 non-null  object 
 8   annual_inc                  933160 non-null  float64
 9   verification_status         933160 non-null  object 
 10  issue_d                     933160 non-null  object 
 11  loan_status                 933160 non-null  object 
 12  purpose                     933160 non-null  object 
 13  zip_code      

,0
loan_amnt,0
funded_amnt,0
int_rate,0
installment,0
grade,0
sub_grade,0
emp_length,58768
home_ownership,0
annual_inc,0
verification_status,0


In [85]:
key_variables = loan[['loan_amnt','int_rate','installment','emp_length','annual_inc','dti','fico_range_low','fico_range_high','revol_bal','revol_util','total_acc','return']]
key_variables.describe()

,loan_amnt,int_rate,installment,annual_inc,dti,fico_range_low,fico_range_high,revol_bal,revol_util,total_acc,return
count,933160.000000,933160.000000,933160.000000,9.331600e+05,933160.000000,933160.000000,933160.000000,9.331600e+05,933160.000000,933160.000000,933160.000000
mean,12559.115559,0.119775,416.318741,7.412200e+04,17.833822,695.139858,699.139987,1.565339e+04,0.523420,24.273673,0.065197
std,8042.750083,0.039952,268.816718,6.938399e+04,8.377159,31.359123,31.359739,2.242787e+04,0.240878,11.821678,0.247727
min,500.000000,0.053200,14.010000,3.000000e+03,-1.000000,660.000000,664.000000,0.000000e+00,0.000000,2.000000,-1.000000
25%,6425.000000,0.089000,215.630000,4.400000e+04,11.520000,670.000000,674.000000,5.705000e+03,0.344000,16.000000,0.073529
50%,10000.000000,0.115300,339.310000,6.200000e+04,17.280000,690.000000,694.000000,1.049600e+04,0.526000,23.000000,0.127677
75%,16275.000000,0.143300,549.940000,9.000000e+04,23.700000,710.000000,714.000000,1.865600e+04,0.708000,31.000000,0.188737
max,40000.000000,0.309900,1584.900000,9.573072e+06,49.960000,845.000000,850.000000,2.904836e+06,8.923000,176.000000,0.581820


# **Data Preprocessing and Feature Engineering**

`mths_since_last_delinq` (months since last delinquency) has missing values, which can indicate either a long period without delinquency or missing information.

To ensure the dataset remains useful for modeling, replacing missing values with the maximum observed delinquency period under the assumption that borrowers with missing values have the longest delinquency-free history

In [86]:
loan['dummy_delinq'] = loan['mths_since_last_delinq'].isnull().astype(int)
max_delinq = loan['mths_since_last_delinq'].max()
loan['mths_since_last_delinq'] = loan['mths_since_last_delinq'].fillna(max_delinq)
loan['inter_delinq'] = loan['dummy_delinq'] * loan['mths_since_last_delinq']

The `emp_length` variable is stored as a categorical string (e.g., "10+ years", "<1 year", "n/a"), which is not directly usable for numerical models.

Standardizing this feature into a numerical that helps improve model accuracy.

In [87]:
def convert_emp_length(emp):
    if pd.isnull(emp) or emp == 'n/a':
        return np.nan
    elif '<' in str(emp):
        return 0
    elif '+' in str(emp):
        return 10
    else:
        return int(str(emp).split()[0])
loan['emp_length'] = loan['emp_length'].apply(convert_emp_length)
loan['emp_length'] = loan['emp_length'].fillna(loan['emp_length'].mean())

# **Categorical Features**

In [88]:
# grade and subgrade
grade_dummies = pd.get_dummies(loan['grade'], prefix='grade', drop_first=True).astype(int)
loan = pd.concat([loan, grade_dummies], axis=1)
loan.drop(columns=['grade', 'sub_grade'], inplace=True)

# homeownership
home_ownership_dummies = pd.get_dummies(loan['home_ownership'], prefix='home', drop_first=False).astype(int)
home_ownership_dummies.drop(columns=["home_ANY", "home_OTHER", "home_NONE"], inplace=True)
loan = pd.concat([loan, home_ownership_dummies], axis=1)
loan.drop(columns=['home_ownership'], inplace=True)

# verification_status
loan['verification_status'] = loan['verification_status'].map({'Not Verified': 0, 'Verified': 1, 'Source Verified': 1})

# purpose
purpose_dummies = pd.get_dummies(loan['purpose'], prefix='purpose', drop_first=True).astype(int)
loan = pd.concat([loan, purpose_dummies], axis=1)
loan.drop(columns=['purpose'], inplace=True)

# earliest_cr_line
loan['earliest_cr_line'] = pd.to_datetime(loan['earliest_cr_line'], format='%b-%Y')
loan['issue_d'] = pd.to_datetime(loan['issue_d'], format='%b-%Y')
loan['length_credit'] = (loan['issue_d'] - loan['earliest_cr_line']).dt.days / 365.25
loan.drop(columns=['earliest_cr_line'], inplace=True)

# fico_range_low and fico_range_high
loan.drop(columns=['fico_range_high'], inplace=True)

loan.drop(columns=['loan_amnt'], inplace=True)

# **Feature Engineering**

Interaction term: Annual income and verification

In [89]:
loan['inter_income'] = loan['verification_status'] * loan['annual_inc']

Creating a new feature `state_income`

In [90]:
state_income = loan[loan['verification_status'] == 1].groupby(['addr_state', 'year'])['annual_inc'].mean().reset_index()
state_income.rename(columns={'annual_inc': 'state_income'}, inplace=True)
loan = loan.merge(state_income, on=['addr_state', 'year'], how='left')
loan['state_income'] = loan['state_income'].fillna(loan['state_income'].mean())

# dropping the location related features
loan.drop(columns=['addr_state','zip_code'], inplace=True)

## Standardizing Features for Consistency in Modeling

Many numerical features in the dataset have different scales (some are in thousands, while others are between 0 and 1).

Applied feature scaling using StandardScaler() to transform features to have mean = 0 and standard deviation = 1.

Grouped by year before scaling → This ensures the scaling is applied year by year, preventing data leakage across different time periods.

In [91]:
scale_fea_set = ['funded_amnt', 'installment', 'emp_length', 'annual_inc',
                'dti', 'delinq_2yrs', 'fico_range_low', 'inq_last_6mths',
                'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal',
                'revol_util', 'total_acc', 'collections_12_mths_ex_med',
                'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt',
                'pub_rec_bankruptcies', 'tax_liens', 'inter_delinq',
                'length_credit', 'inter_income', 'state_income']

scaled_df = pd.DataFrame(columns=loan.columns)

# Using a list to collect scaled dataframes for concatenation
scaled_dataframes = []

for year, data in loan.groupby('year'):
    scaler = StandardScaler()
    data[scale_fea_set] = scaler.fit_transform(data[scale_fea_set])
    scaled_dataframes.append(data)

# Concatenating all scaled dataframes at once
scaled_df = pd.concat(scaled_dataframes, ignore_index=True)

In [92]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(scaled_df.head())

   funded_amnt  int_rate  installment  emp_length  annual_inc  verification_status    issue_d  loan_status       dti  delinq_2yrs  fico_range_low  inq_last_6mths  mths_since_last_delinq  open_acc  pub_rec  revol_bal  revol_util  total_acc  collections_12_mths_ex_med  acc_now_delinq  chargeoff_within_12_mths  delinq_amnt  pub_rec_bankruptcies  tax_liens  year  early_default    return  dummy_delinq  inter_delinq  grade_B  grade_C  grade_D  grade_E  grade_F  grade_G  home_MORTGAGE  home_OWN  home_RENT  purpose_credit_card  purpose_debt_consolidation  purpose_educational  purpose_home_improvement  purpose_house  purpose_major_purchase  purpose_medical  purpose_moving  purpose_other  purpose_renewable_energy  purpose_small_business  purpose_vacation  purpose_wedding  length_credit  inter_income  state_income
0    -1.313520    0.0800    -1.299955   -1.119802   -0.611587                    0 2008-12-01   Fully Paid -0.774628    -0.335851        0.714442       -0.046682                0.818283



---



# **Training Linear Regression Model**

This code implements a linear regression model. The goal is to train the model on past data and evaluate its ability to predict future loan returns.

An **expanding-window approach** was used instead of a **rolling-window approach** to ensure that the model progressively learns from all available historical data.

**Reason:**
*  Each training set includes all past data up to the current validation year.
*  The training dataset grows over time, incorporating more historical patterns.
*  This is beneficial for financial datasets, where more historical data often leads to better trend learning.
*  Better loan return prediction, avoiding unnecessary loss of valuable information.

In [93]:
scaled_df = scaled_df.drop(columns=['issue_d', 'loan_status', 'early_default'], errors='ignore')

# Converting year to numeric if needed
scaled_df['year'] = scaled_df['year'].astype(int)

scaled_df = scaled_df.dropna().reset_index(drop=True)
X = scaled_df.drop(columns=['year', 'return'])
y = scaled_df['return']
years = scaled_df['year'].unique()

# Sorting data by year
scaled_df = scaled_df.sort_values(by='year').reset_index(drop=True)

in_sample_r2 = []
out_sample_r2 = []

# Initializing model once
model = LinearRegression()

for i in range(1, len(years)):
    train_data = scaled_df[scaled_df['year'] <= years[i-1]]
    val_data = scaled_df[scaled_df['year'] == years[i]]

    train_X, train_y = train_data.drop(columns=['year', 'return']), train_data['return']
    val_X, val_y = val_data.drop(columns=['year', 'return']), val_data['return']

    model.fit(train_X, train_y)

    # Predicting and calculating R² scores
    in_sample_r2.append(r2_score(train_y, model.predict(train_X)))
    out_sample_r2.append(r2_score(val_y, model.predict(val_X)))

print(f'Final In-Sample R²: {in_sample_r2[-1]}')
print(f'Final Out-of-Sample R²: {out_sample_r2[-1]}')


Final In-Sample R²: 0.012681313018244933
Final Out-of-Sample R²: 0.0030258989198767017




---



# **Training Logistic Regression Model**

This section implements logistic regression to predict loan status (whether a loan is Fully Paid or Charged Off).

In [94]:
scaled_df['loan_status'] = loan['loan_status'].values
scaled_df['loan_status'] = (scaled_df['loan_status'] == 'Charged Off').astype(int)
X = scaled_df.drop(columns=['issue_d', 'year', 'return', 'early_default', 'loan_status'], errors='ignore')
y = scaled_df['loan_status']
years = scaled_df['year'].unique()

scaled_df = scaled_df.sort_values(by='year').reset_index(drop=True)

accuracy_scores = []
f1_scores = []
auc_scores = []

model = LogisticRegression(solver='liblinear')  # Logistic Regression Model

for i in range(1, len(years)):
    train_data = scaled_df[scaled_df['year'] <= years[i-1]]
    val_data = scaled_df[scaled_df['year'] == years[i]]

    train_X, train_y = train_data.drop(columns=['year', 'loan_status']), train_data['loan_status']
    val_X, val_y = val_data.drop(columns=['year', 'loan_status']), val_data['loan_status']

    # Training model
    model.fit(train_X, train_y)

    # Predicting probabilities and classes
    val_preds = model.predict(val_X)
    val_probs = model.predict_proba(val_X)[:, 1]  # Probability of being 'Charged Off'

    accuracy_scores.append(accuracy_score(val_y, val_preds))
    f1_scores.append(f1_score(val_y, val_preds))
    auc_scores.append(roc_auc_score(val_y, val_probs))

print(f'Final Accuracy: {accuracy_scores[-1]:.4f}')
print(f'Final F1 Score: {f1_scores[-1]:.4f}')
print(f'Final AUC Score: {auc_scores[-1]:.4f}')

Final Accuracy: 0.8422
Final F1 Score: 0.0000
Final AUC Score: 0.5008


**Accuracy (84.22%)** → Model correctly classifies most loans.

**F1 Score (0.0000)** → Indicates poor performance in predicting "Charged Off" loans (due to class imbalance).

**AUC Score (0.5008)** → Suggests the model has little predictive power beyond random guessing



---




Applied **Synthetic Minority Over-sampling Technique (SMOTE)** to handle class imbalance in loan status classification. Before applying SMOTE, the dataset had far fewer "Charged Off" loans compared to "Fully Paid" loans, leading to poor F1 score and AUC performance in logistic regression.

In [95]:
X = scaled_df.drop(columns=['year', 'loan_status'])
y = scaled_df['loan_status']

print("Before Resampling:", Counter(y))

smote = SMOTE(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("After Resampling:", Counter(y_resampled))

Before Resampling: Counter({0: 797918, 1: 135242})
After Resampling: Counter({0: 797918, 1: 797918})


After applying SMOTE to balance the dataset, the logistic regression model was retrained on the resampled data and evaluated on the most recent year's validation set.

In [96]:
# Retraining logistic regression after resampling
model = LogisticRegression(solver='liblinear')

model.fit(X_resampled, y_resampled)

val_X = scaled_df[scaled_df['year'] == scaled_df['year'].max()].drop(columns=['year', 'loan_status'])
val_y = scaled_df[scaled_df['year'] == scaled_df['year'].max()]['loan_status']

val_preds = model.predict(val_X)
val_probs = model.predict_proba(val_X)[:, 1]

# Computing new metrics
accuracy = accuracy_score(val_y, val_preds)
f1 = f1_score(val_y, val_preds)
auc = roc_auc_score(val_y, val_probs)

print(f'Post-Resampling Accuracy: {accuracy:.4f}')
print(f'Post-Resampling F1 Score: {f1:.4f}')
print(f'Post-Resampling AUC Score: {auc:.4f}')

Post-Resampling Accuracy: 0.6593
Post-Resampling F1 Score: 0.1972
Post-Resampling AUC Score: 0.5014


SMOTE successfully improved F1 Score, meaning the model is better at detecting defaults.

Lower accuracy but better balance → The model is no longer over-predicting "Fully Paid" loans.



---



**Optimizing Decision Threshold for Logistic Regression**

After resampling, the logistic regression model still struggled to differentiate defaulters from non-defaulters (low AUC score). To improve classification performance, this step tuned the decision threshold instead of relying on the default 0.5 threshold.



In [97]:
import numpy as np

# Testing different thresholds manually
thresholds = np.linspace(0.1, 0.9, 10)
best_f1 = 0
best_thresh = 0.5

for thresh in thresholds:
    val_preds_opt = (val_probs >= thresh).astype(int)
    f1 = f1_score(val_y, val_preds_opt)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f'Best Threshold Found: {best_thresh:.4f}')

val_preds_final = (val_probs >= best_thresh).astype(int)

# Computing final metrics
accuracy_final = accuracy_score(val_y, val_preds_final)
f1_final = f1_score(val_y, val_preds_final)
auc_final = roc_auc_score(val_y, val_probs)

print(f'Final Accuracy: {accuracy_final:.4f}')
print(f'Final F1 Score: {f1_final:.4f}')
print(f'Final AUC Score: {auc_final:.4f}')


Best Threshold Found: 0.1000
Final Accuracy: 0.1601
Final F1 Score: 0.2726
Final AUC Score: 0.5014


---

**Using Predicted Default Probability as a Feature for Loan Return Prediction**

Aiming to integrate the predicted probability of loan default `(pred_default_prob)` into the regression model for loan returns to capture the relationship between default risk and loan returns, improving predictive accuracy.

In [98]:
X_corrected = scaled_df.drop(columns=['year', 'loan_status', 'return'], errors='ignore')
y_corrected = scaled_df['loan_status']

model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_corrected, y_corrected)

LogisticRegression(random_state=42, solver='liblinear')

In [99]:
scaled_df['pred_default_prob'] = model.predict_proba(X_corrected)[:, 1]

X_new = scaled_df.drop(columns=['year', 'loan_status', 'return'], errors='ignore')  # Features
y_new = scaled_df['return']

model_new = LinearRegression()
model_new.fit(X_new, y_new)

in_sample_r2_new = model_new.score(X_new, y_new)

print(f'Final In-Sample R² with Default Probability: {in_sample_r2_new:.4f}')

Final In-Sample R² with Default Probability: 0.0131


In [100]:
X_new.head()

,funded_amnt,int_rate,installment,emp_length,annual_inc,verification_status,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,dummy_delinq,inter_delinq,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,home_MORTGAGE,home_OWN,home_RENT,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,length_credit,inter_income,state_income,pred_default_prob
0,-1.313520,0.0800,-1.299955,-1.119802,-0.611587,0,-0.774628,-0.335851,0.714442,-0.046682,0.818283,-0.328315,-0.32524,-0.601596,-0.919980,-0.531015,0.0,0.0,0.0,0.0,-0.291514,0.0,1,0.831765,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.177348,-0.418764,0.004114,0.131439
1,0.151291,0.1126,0.138190,1.587845,-0.358143,0,-1.434677,-0.335851,0.009122,1.606767,-0.969049,-0.328315,-0.32524,-0.753858,-1.263421,0.467056,0.0,0.0,0.0,0.0,-0.291514,0.0,0,-1.202262,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,-0.121256,-0.418764,-0.060697,0.118279
2,1.418155,0.1284,1.439106,0.504786,2.999983,1,1.249219,-0.335851,-0.414071,1.606767,0.818283,1.218852,-0.32524,3.980607,1.189237,0.648523,0.0,0.0,0.0,0.0,-0.291514,0.0,1,0.831765,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,-0.635629,4.644944,-0.088626,0.118342
3,0.032523,0.1221,0.043632,1.046315,-0.484865,0,1.427040,-0.335851,-0.414071,0.780043,-0.660445,-0.770363,-0.32524,-0.443400,-0.791623,-0.893950,0.0,0.0,0.0,0.0,-0.291514,0.0,0,-1.202262,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,-0.728520,-0.418764,-1.310638,0.119550
4,-0.957215,0.0932,-0.958115,-1.119802,-0.995402,0,0.845354,-0.335851,0.291250,0.780043,0.818283,1.439876,-0.32524,-0.540499,-1.117719,0.013387,0.0,0.0,0.0,0.0,-0.291514,0.0,1,0.831765,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,-1.190804,-0.418764,-0.801377,0.120275


# **Processing Test Data to Match Training Data**

In [101]:
# Loading test data
loan_test = pd.read_csv('lc_loan_test.csv')

# Apply missing value handling (same as training)
loan_test['dummy_delinq'] = loan_test['mths_since_last_delinq'].isnull().astype(int)
max_delinq = loan_test['mths_since_last_delinq'].max()
loan_test['mths_since_last_delinq'] = loan_test['mths_since_last_delinq'].fillna(max_delinq)
loan_test['inter_delinq'] = loan_test['dummy_delinq'] * loan_test['mths_since_last_delinq']

def convert_emp_length(emp):
    if pd.isnull(emp) or emp == 'n/a':
        return np.nan
    elif '<' in str(emp):
        return 0
    elif '+' in str(emp):
        return 10
    else:
        return int(str(emp).split()[0])

loan_test['emp_length'] = loan_test['emp_length'].apply(convert_emp_length)
loan_test['emp_length'] = loan_test['emp_length'].fillna(loan_test['emp_length'].mean())

# One-hot encoding categorical variables (ensuring test set has the same dummies as training)
grade_dummies = pd.get_dummies(loan_test['grade'], prefix='grade', drop_first=True).astype(int)
loan_test = pd.concat([loan_test, grade_dummies], axis=1)
loan_test.drop(columns=['grade', 'sub_grade'], inplace=True)

home_ownership_dummies = pd.get_dummies(loan_test['home_ownership'], prefix='home', drop_first=False).astype(int)
loan_test = pd.concat([loan_test, home_ownership_dummies], axis=1)
loan_test.drop(columns=['home_ownership'], inplace=True)

loan_test['verification_status'] = loan_test['verification_status'].map({'Not Verified': 0, 'Verified': 1, 'Source Verified': 1})

purpose_dummies = pd.get_dummies(loan_test['purpose'], prefix='purpose', drop_first=True).astype(int)
loan_test = pd.concat([loan_test, purpose_dummies], axis=1)
loan_test.drop(columns=['purpose'], inplace=True)

expected_dummies = [col for col in X_new.columns if 'purpose_' in col or 'grade_' in col or 'home_' in col]
for col in expected_dummies:
    if col not in loan_test.columns:
        loan_test[col] = 0  # Assign 0 if missing

# Feature engineering:
loan_test['earliest_cr_line'] = pd.to_datetime(loan_test['earliest_cr_line'], format='%b-%Y')
loan_test['issue_d'] = pd.to_datetime(loan_test['issue_d'], format='%b-%Y')
loan_test['length_credit'] = (loan_test['issue_d'] - loan_test['earliest_cr_line']).dt.days / 365.25
loan_test.drop(columns=['earliest_cr_line', 'issue_d'], inplace=True)

# Drop unnecessary columns (ensuring same as training)
loan_test.drop(columns=['fico_range_high', 'zip_code'], inplace=True, errors='ignore')

# Creating interaction features:
loan_test['inter_income'] = loan_test['verification_status'] * loan_test['annual_inc']

state_income = loan_test.groupby(['addr_state'])['annual_inc'].mean().reset_index()
state_income.rename(columns={'annual_inc': 'state_income'}, inplace=True)
loan_test = loan_test.merge(state_income, on=['addr_state'], how='left')
loan_test['state_income'] = loan_test['state_income'].fillna(loan_test['state_income'].mean())
loan_test.drop(columns=['addr_state'], inplace=True, errors='ignore')

# Standardize features using the trained StandardScaler per year
scale_fea_set = ['funded_amnt', 'installment', 'emp_length', 'annual_inc',
                'dti', 'delinq_2yrs', 'fico_range_low', 'inq_last_6mths',
                'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal',
                'revol_util', 'total_acc', 'collections_12_mths_ex_med',
                'acc_now_delinq', 'chargeoff_within_12_mths', 'delinq_amnt',
                'pub_rec_bankruptcies', 'tax_liens', 'inter_delinq',
                'length_credit', 'inter_income', 'state_income']

scaled_test_dataframes = []

for year, data in loan_test.groupby('year'):
    scaler = StandardScaler()
    data[scale_fea_set] = scaler.fit_transform(data[scale_fea_set])
    scaled_test_dataframes.append(data)

loan_test_scaled = pd.concat(scaled_test_dataframes, ignore_index=True)

# Ensuring test set has the same features as training
for col in X_new.columns:
    if col not in loan_test_scaled.columns:
        loan_test_scaled[col] = 0  # Add missing columns with default value

X_test = loan_test_scaled.drop(columns=['year', 'id', 'home_ANY', 'loan_amnt','pred_default_prob'])

In [102]:
X_test.head()

,funded_amnt,int_rate,installment,emp_length,annual_inc,verification_status,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,dummy_delinq,inter_delinq,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,home_MORTGAGE,home_OWN,home_RENT,purpose_credit_card,purpose_debt_consolidation,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_educational,purpose_wedding,length_credit,inter_income,state_income
0,-0.273242,0.0749,-0.348363,-0.815644,-0.149072,0,1.057291,-0.373089,1.754870,-0.634009,1.005969,0.240422,-0.376554,-0.333880,-1.209473,-0.367753,-0.13899,-0.073955,-0.087046,-0.022007,-0.362396,-0.160535,1,1.037602,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,-1.135511,-0.27863,-0.874968
1,0.069522,0.1144,0.049565,1.144693,-0.126307,0,-0.888990,-0.373089,1.300833,-0.634009,1.005969,-0.108522,-0.376554,-0.146809,-0.616846,-0.029157,-0.13899,-0.073955,-0.087046,-0.022007,-0.362396,-0.160535,1,1.037602,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,-0.258045,-0.27863,0.236132
2,-1.070168,0.0699,-1.086650,1.144693,0.017871,0,0.350798,-0.373089,-0.061277,-0.634009,1.005969,0.240422,-0.376554,1.347657,0.783909,0.055492,-0.13899,-0.073955,-0.087046,-0.022007,-0.362396,-0.160535,1,1.037602,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0.746560,-0.27863,0.064820
3,-0.387496,0.1139,-0.398231,1.144693,-0.098483,0,0.203768,-0.373089,2.511598,-0.634009,1.005969,0.240422,-0.376554,0.222993,-0.761894,0.140142,-0.13899,-0.073955,-0.087046,-0.022007,-0.362396,-0.160535,1,1.037602,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0.048686,-0.27863,0.441545
4,-0.044732,0.0699,-0.146651,1.144693,-0.118719,0,-0.005563,-0.373089,2.360252,-0.634009,1.005969,0.240422,-0.376554,-0.408511,-1.010549,0.394089,-0.13899,-0.073955,-0.087046,-0.022007,-0.362396,-0.160535,1,1.037602,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0.640958,-0.27863,-0.478139


In [103]:
X_test = loan_test_scaled[X_corrected.columns]

**Predicting Default Probability for Test Data**

In [104]:
pred_default_prob = model.predict_proba(X_test)[:, 1]

In [105]:
print(pred_default_prob)

[0.14804849 0.14745972 0.14518542 ... 0.14814923 0.1489095  0.15159538]


In [106]:
X_test['pred_default_prob'] = pred_default_prob

<ipython-input-106-b60aa85f0b74>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['pred_default_prob'] = pred_default_prob


Predicting loan returns using the trained regression model

In [107]:
loan_test_scaled['return'] = model_new.predict(X_test)

**Saving Final Predictions to `submission_test.csv`**

After predicting loan returns for the test dataset, this step ensures the predictions are correctly formatted and saved in the required submission file.



In [80]:
submission = pd.read_csv('submission_test.csv')

# Ensuring predictions match the correct IDs
submission = submission[['id']]
submission = submission.merge(loan_test_scaled[['id', 'return']], on='id', how='left')

# Saving the updated submission file
submission.to_csv('submission_test.csv', index=False)

print("✅ Predictions successfully saved to submission_test.csv")

✅ Predictions successfully saved to submission_test.csv


# **Final Score**

Percentage of defaults predicted:

# `Score: 0.05316`

## **Use of Generative AI Tools**

In this assignment, generative AI tools such as ChatGPT were used to enhance efficiency and ensure clarity in reporting. Below are specific examples of how AI was utilized:

**Code Explanation & Debugging**
*   Clarified complex machine learning concepts (e.g., the impact of class imbalance and the need for threshold tuning).
*   Assisted in debugging errors related to majorly feature mismatches and prediction issues.
*   Suggested improvements to model implementation using **SMOTE** to balance the dataset.

**Model Optimization Guidance**
*   Recommended techniques such as expanding-window regression over rolling-window approaches, ensuring a more stable model.

**Understanding Regression Outputs**
*   Explained how logistic regression predicts the probability of a loan being "`Charged Off`" (default) versus "`Fully Paid`".
*   Helped interpret Accuracy, F1 Score, AUC Score and why initial F1 and AUC scores were low and suggested solutions like resampling and threshold tuning.

**Assistance with `pred_default_prob` for Test Data**

One of the key challenges in this assignment was generating the `pred_default_prob` (predicted default probability) for the test dataset. Initially, there was an issue where the test data was not aligned with the training data, leading to errors when trying to apply the trained logistic regression model.

Through AI assistance, I was able to identify the error, allowing `pred_default_prob` to be successfully computed for the test dataset.
